# Thử TTS trên Colab — Qwen 1.7B VoiceDesign

Chọn **Runtime → Change runtime type → GPU**, rồi chạy lần lượt. Bài thử dùng văn bản mẫu và giọng tổng hợp, không cần Drive hay giọng cá nhân. Giữ FP32, trọng số và thiết lập gốc. Đo tải model, nạp model và ba lượt sinh audio riêng. GPU tùy phiên; không tự mua compute hay nâng cấp gói.


In [ ]:
import subprocess, sys, time, json, shutil
from pathlib import Path
START = time.perf_counter()
assert shutil.which('nvidia-smi'), 'Chưa có GPU. Chọn GPU trong Runtime > Change runtime type.'
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], check=True)
ROOT = Path('/content/logdd-tts-test')
ROOT.mkdir(parents=True, exist_ok=True)


In [ ]:
setup_start = time.perf_counter()
subprocess.run([sys.executable, '-m', 'pip', 'install', 'uv==0.12.3'], check=True)
subprocess.run([sys.executable, '-m', 'uv', 'venv', '--python', '3.12', str(ROOT / '.venv')], check=True)
PYTHON = str(ROOT / '.venv/bin/python')
subprocess.run([sys.executable, '-m', 'uv', 'pip', 'install', '--python', PYTHON,
    'torch==2.8.0', 'torchaudio==2.8.0', 'qwen-tts==0.1.1',
    'transformers==4.57.3', 'soundfile==0.13.1'], check=True)
SETUP_SECONDS = time.perf_counter() - setup_start
print('Cài môi trường:', round(SETUP_SECONDS, 2), 'giây')


In [ ]:
SCRIPT = '"""Short CUDA TTS probe for Colab. Uses synthetic text, no user audio/files."""\nimport json\nimport time\nfrom pathlib import Path\nimport importlib.metadata\n\nimport numpy as np\nimport soundfile as sf\nimport torch\nfrom huggingface_hub import snapshot_download\nfrom qwen_tts import Qwen3TTSModel\n\nROOT = Path(\'/content/logdd-tts-test\')\nOUT = ROOT / \'results\'\nOUT.mkdir(parents=True, exist_ok=True)\nTEXT = \'Marketing turns attention into trust. Finance turns decisions into long term value.\'\nINSTRUCTION = \'A young adult male voice, clear and friendly, with a natural American accent.\'\nREPO = \'Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign\'\nassert torch.cuda.is_available(), \'Chưa có GPU CUDA. Chọn GPU trong Runtime > Change runtime type.\'\nreport = {\n    \'model\': REPO, \'gpu\': torch.cuda.get_device_name(0),\n    \'precision\': \'float32\', \'attention\': \'eager\',\n    \'text\': TEXT, \'instruction\': INSTRUCTION,\n    \'versions\': {name: importlib.metadata.version(name) for name in (\'torch\', \'qwen-tts\', \'transformers\')},\n    \'runs\': [],\n}\nt = time.perf_counter()\nmodel_dir = snapshot_download(REPO)\nreport[\'download_seconds\'] = time.perf_counter() - t\nt = time.perf_counter()\nmodel = Qwen3TTSModel.from_pretrained(model_dir, device_map=\'cuda:0\', dtype=torch.float32,\n                                     attn_implementation=\'eager\')\ntorch.cuda.synchronize()\nreport[\'load_seconds\'] = time.perf_counter() - t\n# Keep original weights, sampling, token limit and precision for this first test.\n# GPU and MLX random generation need not produce identical voices or duration.\nfor i in range(3):\n    torch.manual_seed(123 + i)\n    torch.cuda.reset_peak_memory_stats()\n    torch.cuda.synchronize()\n    t = time.perf_counter()\n    waves, sr = model.generate_voice_design(text=TEXT, language=\'English\', instruct=INSTRUCTION)\n    torch.cuda.synchronize()\n    seconds = time.perf_counter() - t\n    wave = np.asarray(waves[0], dtype=np.float32)\n    assert wave.size and np.isfinite(wave).all(), \'Model returned empty or invalid audio\'\n    filename = f\'qwen-cuda-{i + 1}.wav\'\n    sf.write(OUT / filename, wave, sr)\n    row = dict(run=i + 1, kind=\'first\' if i == 0 else \'warm\', generation_seconds=seconds,\n               audio_seconds=len(wave) / sr, rtf=seconds / (len(wave) / sr),\n               sample_rate=sr, peak_gpu_gb=torch.cuda.max_memory_allocated() / 1e9, file=filename)\n    report[\'runs\'].append(row)\n    (OUT / \'benchmark.json\').write_text(json.dumps(report, indent=2), encoding=\'utf-8\')\n    print(json.dumps(row), flush=True)\nprint(\'TTS_TEST_COMPLETE\', flush=True)\n'
(ROOT / 'qwen_benchmark.py').write_text(SCRIPT, encoding='utf-8')
subprocess.run([PYTHON, '-u', str(ROOT / 'qwen_benchmark.py')], check=True, timeout=1200)


In [ ]:
from IPython.display import Audio, display
OUT = ROOT / 'results'
report = json.loads((OUT / 'benchmark.json').read_text())
report['setup_seconds'] = SETUP_SECONDS
report['total_before_download_seconds'] = time.perf_counter() - START
(OUT / 'benchmark.json').write_text(json.dumps(report, indent=2), encoding='utf-8')
print(json.dumps(report, indent=2))
display(Audio(filename=str(OUT / 'qwen-cuda-3.wav')))


In [ ]:
from google.colab import files
archive = shutil.make_archive(str(ROOT / 'tts-results'), 'zip', OUT)
print('Bắt đầu tải WAV và số đo về máy. Thời gian tải file cần đo riêng ở máy nhận.')
files.download(archive)
